# Time-Slicing for GPU Sharing

A practical reference for **GPU time-slicing** on NVIDIA hardware in Kubernetes — how it
oversubscribes a single physical GPU across many pods, how to configure it through the
NVIDIA GPU Operator / `k8s-device-plugin`, and when to reach for it instead of MIG or MPS.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

<a id='introduction'></a>

### What is it?

**Time-slicing** lets multiple CUDA contexts share one physical GPU by *interleaving* their
work on the GPU's compute engines over time. The GPU's hardware scheduler rapidly switches
between the contexts — each gets the **whole** GPU for a short slice, then yields to the next.
This is the same time-sharing the driver already does for two processes on your laptop; what
the NVIDIA Kubernetes device plugin adds is the ability to **advertise one GPU as N
"replicas"** so the scheduler will place up to N pods on it.

Concretely, instead of the node reporting `nvidia.com/gpu: 1`, with a replica count of 4 it
reports `nvidia.com/gpu: 4`. Four pods can each request `nvidia.com/gpu: 1` and all land on
the same physical device.

### Why use it?

- **Higher utilization / lower cost.** Most inference, notebook, and dev workloads use a tiny
  fraction of an A100/H100. Time-slicing packs many of them onto one card instead of
  stranding an entire GPU per pod.
- **Works on every NVIDIA GPU.** Unlike MIG, time-slicing needs no special hardware support —
  it runs on Pascal, Volta, Turing, Ampere, Hopper, and on consumer cards (T4, L4, RTX).
- **Zero application changes.** Pods request GPUs exactly as before; sharing is transparent to
  CUDA code.
- **Trivial to enable.** A single ConfigMap toggles it on, no node reboot required.

### When to use it?

- Bursty or low-duty-cycle workloads: model-serving replicas, Jupyter notebooks, CI test jobs.
- Development / staging clusters where many engineers share a few GPUs.
- Cards that **do not** support MIG (anything pre-Ampere, or Ampere consumer GPUs).
- Situations where you accept **no memory or fault isolation** between tenants (see Pitfalls).

## Key Features

<a id='key-features'></a>

### Core Capabilities of GPU Time-Slicing

| Feature | Description | Benefit |
|---------|-------------|---------|
| Oversubscription via `replicas` | Each physical GPU is advertised as *N* schedulable units | Pack many pods onto one card without code changes |
| Hardware-scheduled context switch | The GPU rotates compute engines between CUDA contexts | No per-job static partition; idle slices are reclaimed |
| `renameByDefault` option | Exposes the shared resource as `nvidia.com/gpu.shared` | Workloads can *opt in* to sharing vs. demanding a whole GPU |
| Per-node config via GFD labels | Different replica counts on different node pools | A100 nodes share 8-way, T4 nodes 2-way, from one ConfigMap |
| Works alongside the GPU Operator | Enabled by `devicePlugin.config` in the ClusterPolicy/Helm values | Declarative, GitOps-friendly rollout |
| No hardware prerequisite | Independent of MIG; runs on any CUDA GPU | Universal coverage, including consumer cards |

## Architecture Overview

<a id='architecture'></a>

```
                Kubernetes node (1 physical GPU)
  +-----------------------------------------------------------+
  |  kubelet  <-- Allocate() --  nvidia-device-plugin         |
  |                                 |   reads ConfigMap:      |
  |                                 |   replicas: 4           |
  |   advertises  nvidia.com/gpu: 4 (4 schedulable units)     |
  |                                                           |
  |   Pod A   Pod B   Pod C   Pod D   (each requests 1)       |
  |     \       |       |       /                             |
  |      \      |       |      /                              |
  |   +--------------------------------+                      |
  |   |   NVIDIA driver / GPU HW       |  time-slices the     |
  |   |   compute engines (SMs)        |  4 CUDA contexts     |
  |   +--------------------------------+                      |
  |        single 40GB framebuffer (SHARED, not partitioned)  |
  +-----------------------------------------------------------+
```

### Components

1. **NVIDIA Kubernetes device plugin** — the piece that actually implements time-slicing. It
   reads a *sharing config* and, for each GPU, registers `replicas` copies of the same device
   ID with kubelet. It does **not** enforce memory or compute limits; it only changes how many
   allocatable units kubelet sees.
2. **GPU Feature Discovery (GFD)** — labels nodes with GPU model/driver info and (when sharing
   is on) with `nvidia.com/gpu.replicas` and `nvidia.com/gpu.product=...-SHARED` so you can
   target shared nodes with `nodeSelector`.
3. **NVIDIA GPU Operator** — packages the device plugin + GFD + driver + DCGM and exposes the
   time-slicing config through a single `ClusterPolicy`/Helm value (`devicePlugin.config`).
4. **GPU hardware scheduler** — the on-die scheduler that interleaves the CUDA contexts. This
   is what makes the sharing real; the Kubernetes layer only does the bookkeeping.

## Installation

<a id='installation'></a>

### Prerequisites

- A Kubernetes cluster with NVIDIA GPU nodes and the NVIDIA driver + container toolkit
  installed (or let the GPU Operator install them).
- `kubectl` and `helm` v3.
- The **NVIDIA GPU Operator** (recommended) or a standalone `nvidia-device-plugin` DaemonSet.

### Installation Steps

Install the GPU Operator via Helm (it bundles the device plugin and GFD that do the slicing):

In [ ]:
# Run these in a shell with kubectl/helm pointed at your cluster.
# (Shown as a Python string so the notebook stays inspectable without a cluster.)
install = r'''
helm repo add nvidia https://helm.ngc.nvidia.com/nvidia
helm repo update

helm install --wait gpu-operator nvidia/gpu-operator \
  --namespace gpu-operator --create-namespace \
  --version v24.9.0
'''
print(install)

## Basic Usage

<a id='basic-usage'></a>

### Quick Start Example

Time-slicing is turned on with a **ConfigMap** that the device plugin reads. The minimal
config replicates every GPU 4 ways:

```yaml
# time-slicing-config.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: time-slicing-config
  namespace: gpu-operator
data:
  any: |-                       # 'any' = applied to all GPU models on the node
    version: v1
    sharing:
      timeSlicing:
        resources:
          - name: nvidia.com/gpu
            replicas: 4         # advertise each physical GPU as 4 units
```

Apply it and point the GPU Operator at it (the operator restarts the device plugin for you):

```bash
kubectl apply -f time-slicing-config.yaml

kubectl patch clusterpolicy/cluster-policy \
  -n gpu-operator --type merge \
  -p '{"spec":{"devicePlugin":{"config":{"name":"time-slicing-config","default":"any"}}}}'
```

After the device plugin rolls out, the node's allocatable GPU count is multiplied by 4:

```bash
kubectl get nodes -o json \
  | jq '.items[].status.allocatable["nvidia.com/gpu"]'
# "4"   <- one physical GPU now schedulable 4 ways
```

Now a normal pod spec schedules onto a slice — nothing GPU-share-specific in the workload:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: cuda-vectoradd
spec:
  restartPolicy: OnFailure
  containers:
    - name: cuda
      image: nvcr.io/nvidia/k8s/cuda-sample:vectoradd-cuda11.7.1-ubi8
      resources:
        limits:
          nvidia.com/gpu: 1   # one *slice*, not one physical GPU
```

In [ ]:
# A tiny helper that renders a time-slicing ConfigMap from a per-model replica map.
# Useful for generating GitOps manifests for heterogeneous node pools.
import yaml  # pip install pyyaml

def time_slicing_configmap(per_model_replicas, name='time-slicing-config',
                           namespace='gpu-operator', rename_by_default=False):
    data = {}
    for model, replicas in per_model_replicas.items():
        data[model] = yaml.safe_dump({
            'version': 'v1',
            'sharing': {
                'timeSlicing': {
                    'renameByDefault': rename_by_default,
                    'resources': [
                        {'name': 'nvidia.com/gpu', 'replicas': replicas},
                    ],
                }
            },
        }, sort_keys=False)
    return yaml.safe_dump({
        'apiVersion': 'v1', 'kind': 'ConfigMap',
        'metadata': {'name': name, 'namespace': namespace},
        'data': data,
    }, sort_keys=False)

# A100 nodes share 8-way, T4 nodes 2-way; 'any' is the fallback.
print(time_slicing_configmap({
    'any': 2,
    'a100-40gb': 8,
    't4': 2,
}))

## Advanced Features

<a id='advanced-features'></a>

### `renameByDefault`: opt-in sharing

By default a sliced GPU is still advertised as `nvidia.com/gpu`, so *every* pod silently lands
on a shared device. Set `renameByDefault: true` and the shared units are instead exposed as
**`nvidia.com/gpu.shared`**. Workloads that can tolerate sharing request
`nvidia.com/gpu.shared: 1`, while latency-critical jobs keep requesting `nvidia.com/gpu: 1`
and get a whole card. This makes sharing an explicit, auditable choice.

```yaml
data:
  any: |-
    version: v1
    sharing:
      timeSlicing:
        renameByDefault: true            # exposes nvidia.com/gpu.shared
        failRequestsGreaterThanOne: true # reject limits > 1 on a shared resource
        resources:
          - name: nvidia.com/gpu
            replicas: 4
```

### `failRequestsGreaterThanOne`

A request like `nvidia.com/gpu: 2` against a sliced resource is **meaningless** — two slices of
the same physical GPU give no extra memory or throughput, just two contexts on one card. Set
`failRequestsGreaterThanOne: true` so the plugin rejects such requests instead of silently
handing out two slices of the same device.

### Per-node configuration with multiple named configs

A single ConfigMap can hold several named entries (`a100`, `t4`, `shared-8`, ...). You then
label nodes to pick which one applies, instead of using one global `default`:

```bash
kubectl label node gpu-node-01 \
  nvidia.com/device-plugin.config=shared-8 --overwrite
```

This is how you run different replica counts across heterogeneous hardware from one source of
truth.

In [ ]:
# Render a 'renameByDefault' config that also rejects oversized requests.
import yaml

cfg = {
    'version': 'v1',
    'sharing': {
        'timeSlicing': {
            'renameByDefault': True,
            'failRequestsGreaterThanOne': True,
            'resources': [{'name': 'nvidia.com/gpu', 'replicas': 4}],
        }
    },
}
print(yaml.safe_dump(cfg, sort_keys=False))
print('Workloads now request:  nvidia.com/gpu.shared: 1')

## Use Cases

<a id='use-cases'></a>

#### Use Case 1: Inference fleet consolidation

- **Context:** 20 small model-serving replicas, each using ~3 GB and <15% SM utilization on an
  A100-40GB. One GPU per replica would need 20 GPUs.
- **Implementation:** `replicas: 8` with `renameByDefault: true`; servers request
  `nvidia.com/gpu.shared: 1`. ~3 A100s now host all 20 replicas.
- **Result:** ~6x fewer GPUs, dramatically lower cost; tail latency rises modestly under
  concurrent bursts (acceptable for these SLAs).

#### Use Case 2: Shared data-science / notebook cluster

- **Context:** 30 engineers each want a JupyterLab with a GPU; most sit idle while a person
  thinks or edits code.
- **Implementation:** `replicas: 4` on the notebook node pool; JupyterHub profiles request one
  slice each.
- **Result:** A 4-GPU node serves up to 16 concurrent notebooks. Because notebooks are bursty,
  perceived performance stays good despite oversubscription.

#### Use Case 3: CI / batch test parallelism

- **Context:** GPU unit tests that each run for seconds and barely touch the device.
- **Implementation:** `replicas: 8` so eight CI pods run per GPU.
- **Result:** Test throughput scales with concurrency, not with GPU count — far cheaper CI.

## Best Practices

<a id='best-practices'></a>

1. **Use `renameByDefault: true` in any multi-tenant cluster.** Making sharing opt-in
   (`nvidia.com/gpu.shared`) prevents latency-sensitive jobs from accidentally landing on a
   shared card.
2. **Set `failRequestsGreaterThanOne: true`.** It turns a silently-wrong request into a clear
   scheduling failure.
3. **Right-size `replicas` to the workload's memory footprint, not just its compute.** Memory
   is *not* partitioned: N slices share one framebuffer. If each pod needs 8 GB on a 40 GB
   card, `replicas` above ~4-5 will cause OOMs no matter how idle the SMs are.
4. **Keep replica counts per node pool.** Use named configs + node labels so A100s and T4s get
   appropriate ratios instead of one global number.
5. **Pair with memory guardrails.** Set `PYTORCH_CUDA_ALLOC_CONF` / TensorFlow memory limits
   or use a soft MPS limit so one tenant can't grab the whole framebuffer.
6. **Monitor per-process memory and SM occupancy with DCGM** before pushing replica counts up.
7. **Don't time-slice latency-critical, full-GPU training.** Use whole GPUs or MIG for those.

## Common Pitfalls

<a id='pitfalls'></a>

1. **No memory isolation — OOMs.** All slices share one framebuffer with **no per-slice
   limit**. One pod allocating a big tensor can OOM every other pod on the GPU. *Avoid by*
   sizing `replicas` to memory and adding application-level memory caps.
2. **No fault isolation.** A pod that triggers an XID error, an ECC fault, or wedges the device
   takes down *all* co-located slices. *Avoid by* not co-scheduling critical workloads.
3. **No performance guarantee / noisy neighbors.** Time-slicing gives fair-ish *time* but no
   QoS; a compute-heavy neighbor inflates everyone's latency. *Avoid by* reserving whole GPUs
   or MIG for SLA-bound serving.
4. **Misreading `replicas` as real capacity.** `nvidia.com/gpu: 4` does **not** mean 4x memory
   or 4x FLOPs — it is the same one GPU, time-shared. *Avoid by* educating users / using
   `gpu.shared` naming.
5. **Requesting more than one slice.** `nvidia.com/gpu: 2` just gives two contexts on the same
   card. *Avoid by* setting `failRequestsGreaterThanOne: true`.
6. **Forgetting it's mutually exclusive with MIG on the same GPU.** A GPU is either in MIG mode
   (partitioned) or time-sliced — you can time-slice *within* a MIG instance, but not run both
   strategies on the same bare GPU resource.

## Performance Optimization

<a id='performance'></a>

### Configuration Tuning

Key parameters to optimize:

- **`replicas`** — the oversubscription ratio. Start low (2) and raise it while watching SM
  occupancy and framebuffer headroom. The sweet spot is where the GPU stays busy but memory
  never saturates.
- **Per-process memory caps** — `PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:...`,
  `TF_FORCE_GPU_ALLOW_GROWTH=true`, or a hard `torch.cuda.set_per_process_memory_fraction()` —
  the practical substitute for the hardware memory isolation time-slicing lacks.
- **Batch size / concurrency per pod** — smaller batches yield to neighbors more often,
  smoothing tail latency across slices.
- **MPS as a complement** — enabling MPS *under* time-slicing lets contexts run truly
  concurrently (spatial sharing) instead of purely interleaved, improving aggregate throughput
  for many small kernels.

In [ ]:
# Estimate a safe replica count from per-pod memory need and GPU framebuffer.
def safe_replicas(gpu_mem_gb, per_pod_mem_gb, headroom=0.85):
    """Replicas that keep total demand under `headroom` of framebuffer."""
    usable = gpu_mem_gb * headroom
    return max(1, int(usable // per_pod_mem_gb))

for pod_mem in (1, 3, 5, 8):
    r = safe_replicas(40, pod_mem)   # A100-40GB
    print(f'pod needs {pod_mem:>2} GB -> safe replicas on A100-40GB: {r}')

# Note: this bounds the *memory* dimension only. Compute oversubscription can be higher
# for bursty/low-duty workloads, but memory OOM is the hard wall with time-slicing.

## Production Deployment

<a id='deployment'></a>

### GitOps with the GPU Operator (recommended)

Bake the sharing config straight into the operator's Helm values so it is version-controlled
and reapplied on every sync:

```yaml
# values.yaml for the gpu-operator chart
devicePlugin:
  config:
    name: time-slicing-config   # the ConfigMap below
    default: any

# Ship the ConfigMap as a separate manifest in the same Helm release / Kustomize base:
# (operator watches it and restarts the device-plugin DaemonSet on change)
```

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: time-slicing-config
  namespace: gpu-operator
data:
  any: |-
    version: v1
    sharing:
      timeSlicing:
        renameByDefault: true
        failRequestsGreaterThanOne: true
        resources:
          - name: nvidia.com/gpu
            replicas: 4
```

### Standalone device plugin (no operator)

If you run the plugin directly, mount the same config and pass `--config-file`:

```bash
helm upgrade -i nvdp nvdp/nvidia-device-plugin \
  --namespace nvidia-device-plugin --create-namespace \
  --set-file config.map.config=time-slicing-config.yaml
```

### Targeting shared nodes

With `renameByDefault`, GFD labels shared nodes `nvidia.com/gpu.product: A100-SXM4-40GB-SHARED`.
Use a `nodeSelector` or affinity to keep full-GPU jobs off them and shared jobs on them.

## Monitoring and Observability

<a id='monitoring'></a>

### Key Metrics to Track

- **`DCGM_FI_DEV_GPU_UTIL` / `DCGM_FI_PROF_SM_ACTIVE`** — is the shared GPU actually busy?
  Low values mean you can raise `replicas`; pegged at 100% with rising latency means you've
  over-packed.
- **`DCGM_FI_DEV_FB_USED` / `DCGM_FI_DEV_FB_FREE`** — framebuffer headroom. This is the metric
  that predicts time-slicing OOMs; alert when free memory drops below one pod's footprint.
- **`DCGM_FI_DEV_GPU_TEMP` / `DCGM_FI_DEV_POWER_USAGE`** — sustained sharing pushes thermals
  and power; watch for throttling.
- **Per-pod latency / queue depth** (from the application) — the real signal that oversubscription
  is hurting users.

### Tooling

- The GPU Operator ships **DCGM Exporter**, which exposes the metrics above to Prometheus.
- Grafana's *NVIDIA DCGM Exporter Dashboard* visualizes per-GPU utilization and memory.
- `nvidia-smi` on the node shows every co-located process and its memory — the quickest way to
  spot a memory hog among slices.

### Logging Best Practices

- Log the **node + GPU UUID** each pod was placed on so you can correlate an OOM/XID with its
  noisy neighbor.
- Emit per-process peak memory at job end to feed back into `replicas` sizing.
- Surface device-plugin and GFD logs (`kubectl logs -n gpu-operator ds/nvidia-device-plugin-daemonset`)
  when allocatable counts look wrong.

## Troubleshooting

<a id='troubleshooting'></a>

#### Issue 1: Allocatable GPU count didn't multiply

**Symptoms:** Node still shows `nvidia.com/gpu: 1` after applying the config.

**Cause:** The device plugin didn't pick up the ConfigMap — wrong namespace/name, the
ClusterPolicy `devicePlugin.config` not patched, or the DaemonSet didn't restart.

**Solution:** Confirm the ConfigMap is in `gpu-operator`, re-check the patch, and verify the
plugin restarted: `kubectl rollout status ds/nvidia-device-plugin-daemonset -n gpu-operator`.
Inspect plugin logs for `time-slicing` config lines.

#### Issue 2: Pods OOM / 'CUDA out of memory' under load

**Symptoms:** Co-located pods crash with OOM even though SM utilization is low.

**Cause:** Time-slicing shares one framebuffer with no per-slice memory cap; combined demand
exceeds GPU memory.

**Solution:** Lower `replicas`, add application memory limits
(`torch.cuda.set_per_process_memory_fraction`), or move memory-heavy jobs to MIG / whole GPUs.

#### Issue 3: `nvidia.com/gpu.shared` not appearing

**Symptoms:** After `renameByDefault: true`, pods requesting `nvidia.com/gpu.shared` stay
`Pending`.

**Cause:** The config wasn't applied to that node's model entry, or GFD hasn't relabeled yet.

**Solution:** Check `kubectl get node <n> -o json | jq '.status.allocatable'` for the
`.shared` resource and confirm the node's `nvidia.com/device-plugin.config` label matches a
config that sets `renameByDefault`.

## Comparison with Alternatives

<a id='comparison'></a>

### How Time-Slicing Compares to Other GPU-Sharing Solutions

| Dimension | Time-Slicing | MIG (Multi-Instance GPU) | MPS (Multi-Process Service) |
|-----------|--------------|--------------------------|------------------------------|
| Sharing model | Temporal — contexts interleave on full GPU | Spatial — GPU split into isolated instances | Spatial — concurrent kernels, one scheduler |
| Memory isolation | **None** (shared framebuffer) | **Hard** (per-instance memory) | Soft (optional per-client cap) |
| Fault isolation | None | **Strong** (instance-level) | Weak (shared context) |
| Hardware needed | Any NVIDIA GPU | Ampere+ data-center (A100/A30/H100) | Volta+ |
| Granularity | `replicas: N` per GPU | Fixed profiles (1g.5gb, 3g.20gb, ...) | Per-client % via env vars |
| Setup cost | Trivial (ConfigMap) | Requires MIG mode + reconfig | Daemon per GPU |
| Best for | Bursty/low-duty inference, notebooks, CI | Multi-tenant prod serving needing isolation | Many small kernels from cooperating procs |

They are **composable**: you can time-slice *inside* a MIG instance, and run MPS *under*
time-slicing to get concurrent (not just interleaved) execution.

### When to Choose Time-Slicing

- Your GPUs **don't support MIG**, or you're on consumer cards (T4, L4, RTX).
- Workloads are bursty / low-duty and you value utilization over isolation.
- You want the **simplest possible** path to packing more pods per GPU.
- You can tolerate no hard memory/fault isolation between tenants.

## Resources

<a id='resources'></a>

### Official Documentation

- GPU Operator — Time-Slicing GPUs: <https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/gpu-sharing.html>
- NVIDIA k8s-device-plugin (sharing config): <https://github.com/NVIDIA/k8s-device-plugin#shared-access-to-gpus>
- GPU Feature Discovery: <https://github.com/NVIDIA/gpu-feature-discovery>

### Tutorials and Guides

- NVIDIA blog — *Improving GPU Utilization in Kubernetes*: <https://developer.nvidia.com/blog/improving-gpu-utilization-in-kubernetes/>
- DCGM Exporter (monitoring shared GPUs): <https://github.com/NVIDIA/dcgm-exporter>
- MPS docs (complementary sharing): <https://docs.nvidia.com/deploy/mps/index.html>

### Related Technologies

- **Multi-Instance GPU (MIG)** — hardware-partitioned, isolated GPU slices.
- **Multi-Process Service (MPS)** — concurrent kernel execution from multiple processes.
- **NVIDIA GPU Operator** — the umbrella that deploys and configures all of the above.